In [ ]:
!pip install lifetimes xgboost lightgbm catboost shap --quiet

In [2]:
# CELL 1 — Environment Setup & Logging
#
# UPGRADED: drive.mount() is called FIRST before setting CLV_BASE_DIR.
# Previous version set the env var before mounting — the Drive path did
# not exist yet at that point, which could cause RotatingFileHandler to
# fail silently on the unmounted path.
# =============================================================================
import os, sys, random
import numpy as np
 
# UPGRADED: Mount Drive FIRST — the CLV_BASE_DIR path must exist before
# any module imports try to resolve LOG_FILE or directory paths.
from google.colab import drive
drive.mount('/content/drive')
 
# Now set env var — Drive is mounted, path is accessible
os.environ["CLV_BASE_DIR"] = "/content/drive/MyDrive/clv-prediction-engine"
 
# Global reproducibility — set before any library imports
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
 
# Add project root to path for src.* imports
project_root = os.environ["CLV_BASE_DIR"]
if project_root not in sys.path:
    sys.path.insert(0, project_root)
 
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")
 
# setup_logging() called after drive.mount() — RotatingFileHandler can now
# safely open LOG_FILE on the mounted Drive path.
from src.config import setup_logging
setup_logging()
print("Logging initialised.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/clv-prediction-engine
Logging initialised.


In [3]:
# =============================================================================
# CELL 2 — Directory Setup & Module Imports (unchanged)
# =============================================================================
from src.config import setup_directories, FEATURE_COLS, SPLIT_DAYS
 
setup_directories()
 
try:
    from src.data_ingestion import load_data, clean_data
    from src.feature_engineering import build_hybrid_features
    from src.modeling import train_and_benchmark, tune_champion_model
    from src.evaluation import evaluate_and_plot, save_model
    print("All modules imported successfully.")
except ModuleNotFoundError as e:
    raise SystemExit(
        f"FATAL: Module import failed — {e}. "
        "Check sys.path and src/__init__.py."
    )

All modules imported successfully.


In [4]:
# =============================================================================
# CELL 3 — Data Loading, Feature Engineering, Modeling, Evaluation
# ONE CHANGE ONLY: compute segment_thresholds from y_train_raw and pass to
# evaluate_and_plot() so segment boundaries are stable (not test-set-derived).
# Everything else identical to your original Cell 3.
# =============================================================================

# Step 1: Load raw data
raw_df   = load_data()

# Step 2: Clean data
clean_df = clean_data(raw_df)

# Step 3: Feature Engineering + Log1p Transform
# UPGRADED: returns 5 values — unpack y_test_raw separately
X_train, X_test, y_train, y_test, y_test_raw = build_hybrid_features(
    clean_df,
    raw_df=raw_df,
    split_days=SPLIT_DAYS,
)

print(f"\ny_train (log-scale) — mean: {y_train.mean():.3f}, max: {y_train.max():.3f}")
print(f"y_test  (log-scale) — mean: {y_test.mean():.3f},  max: {y_test.max():.3f}")
print(f"y_test_raw (dollars) — mean: ${y_test_raw.mean():,.2f}, max: ${y_test_raw.max():,.2f}")

# FIX E-1: Compute segment thresholds from y_train_raw (dollar scale) so
# Top/Mid/Low segment boundaries are fixed and do not shift between runs.
# y_train_raw is recovered by inverting the log1p transform on y_train.
import numpy as np
y_train_raw_recovered  = np.expm1(y_train.values)
positive_train         = y_train_raw_recovered[y_train_raw_recovered > 0]
segment_thresholds     = (
    float(np.percentile(positive_train, 20)),
    float(np.percentile(positive_train, 80)),
)
print(f"Segment thresholds (train-derived) — p20: ${segment_thresholds[0]:,.2f} | p80: ${segment_thresholds[1]:,.2f}")

# Steps 4 & 5: Benchmark + Tune
best_model, champ_name, leaderboard = train_and_benchmark(
    X_train, X_test, y_train, y_test
)
tuned_model = tune_champion_model(
    best_model, champ_name,
    X_train, y_train,
    X_test, y_test,
)

# Steps 6 & 7: Evaluate + Save
# FIX E-1: segment_thresholds passed so segment boundaries are train-derived
evaluate_and_plot(tuned_model, champ_name, X_test, y_test, y_test_raw,
                  segment_thresholds=segment_thresholds)
save_model(tuned_model, FEATURE_COLS)


y_train (log-scale) — mean: 3.667, max: 12.035
y_test  (log-scale) — mean: 3.556,  max: 10.193
y_test_raw (dollars) — mean: $690.22, max: $26,720.70
Segment thresholds (train-derived) — p20: $236.64 | p80: $1,373.14


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)



   MASTER MODEL LEADERBOARD v2.4.0 — Ranked by CV MAE (log-scale)
   Primary metrics: Log-scale | Business metrics: Dollar-scale
   * = Selection ineligible (reference baseline only)
                      Model  Log_MAE  Log_R2  Dollar_MAE   Dollar_R2      WAPE  CV_MAE_Mean  CV_MAE_Std
  Two-Stage (Random Forest)   2.0788 -0.0486    479.9134      0.5720   69.5304       2.1534      0.0657
        Two-Stage (XGBoost)   2.3529 -0.1731    523.5366      0.4743   75.8506       2.2540      0.1120
       Two-Stage (LightGBM)   2.3505 -0.1756    558.5489      0.2204   80.9232       2.3760      0.0887
* BTYD Statistical Baseline   2.7549 -0.1414    515.4034      0.4786   74.6722       2.6559      1.3932
                    XGBoost   2.4630  0.2471    526.4376      0.3888   76.2709       3.0111      0.3051
                 Ridge (L2)   2.5119  0.2498   9641.9749 -13812.7760 1396.9404       3.0127      0.3136
              Random Forest   2.4658  0.2779    507.8813      0.5131   73.5824       3.0

In [5]:
import pandas as pd
from pathlib import Path
from src.config import GRAPHS_DIR
 
print("\nPipeline Execution Complete.")
print(f"Champion Model: {champ_name}\n")
 
if leaderboard.empty:
    print("WARNING: Leaderboard is empty — all models failed.")
else:
    # Primary leaderboard — ranked by CV MAE (log-scale)
    print("=== PRIMARY LEADERBOARD (log-scale — model optimisation view) ===")
    display(
        leaderboard[[
            'Model', 'Log_MAE', 'Log_R2',
            'CV_MAE_Mean', 'CV_MAE_Std'
        ]]
        .head(12)
        .style.format({
            'Log_MAE':     '{:.4f}',
            'Log_R2':      '{:.4f}',
            'CV_MAE_Mean': '{:.4f}',
            'CV_MAE_Std':  '{:.4f}',
        })
        .highlight_min(subset=['CV_MAE_Mean'], color='lightgreen')
        .highlight_max(subset=['Log_R2'],      color='lightyellow')
    )
 
    print("\n=== BUSINESS LEADERBOARD (dollar-scale — recruiter-facing view) ===")
    display(
        leaderboard[[
            'Model', 'Dollar_MAE', 'Dollar_R2', 'WAPE', 'SMAPE'
        ]]
        .head(12)
        .style.format({
            'Dollar_MAE': '${:,.2f}',
            'Dollar_R2':  '{:.4f}',
            'WAPE':       '{:.2f}%',
            'SMAPE':      '{:.2f}%',
        })
        .highlight_min(subset=['Dollar_MAE'], color='lightgreen')
        .highlight_max(subset=['Dollar_R2'],  color='lightyellow')
    )
 
# Segment metrics
seg_path = GRAPHS_DIR / 'segment_metrics.csv'
if seg_path.exists():
    seg_df = pd.read_csv(seg_path)
    print("\n=== SEGMENT-LEVEL PERFORMANCE ===")
    display(
        seg_df.style.format({
            'RMSE':       '${:,.2f}',
            'MAE':        '${:,.2f}',
            'R2':         '{:.4f}',
            'WAPE%':      '{:.2f}%',
            'SMAPE%':     '{:.2f}%',
            'Avg_Actual': '${:,.2f}',
            'Avg_Pred':   '${:,.2f}',
        })
        .highlight_max(subset=['R2'],    color='lightyellow')
        .highlight_min(subset=['WAPE%'], color='lightgreen')
    )
 


Pipeline Execution Complete.
Champion Model: Two-Stage (Random Forest)

=== PRIMARY LEADERBOARD (log-scale — model optimisation view) ===


,Model,Log_MAE,Log_R2,CV_MAE_Mean,CV_MAE_Std
0,Two-Stage (Random Forest),2.0788,-0.0486,2.1534,0.0657
1,Two-Stage (XGBoost),2.3529,-0.1731,2.2540,0.1120
2,Two-Stage (LightGBM),2.3505,-0.1756,2.3760,0.0887
3,BTYD Statistical Baseline,2.7549,-0.1414,2.6559,1.3932
4,XGBoost,2.4630,0.2471,3.0111,0.3051
5,Ridge (L2),2.5119,0.2498,3.0127,0.3136
6,Random Forest,2.4658,0.2779,3.0203,0.3125
7,Linear Regression,2.5141,0.2452,3.0234,0.3364
8,ElasticNet,2.5084,0.2722,3.0331,0.3250
9,LightGBM,2.4719,0.2102,3.0589,0.2918



=== BUSINESS LEADERBOARD (dollar-scale — recruiter-facing view) ===


,Model,Dollar_MAE,Dollar_R2,WAPE,SMAPE
0,Two-Stage (Random Forest),$479.91,0.5720,69.53%,83.21%
1,Two-Stage (XGBoost),$523.54,0.4743,75.85%,94.95%
2,Two-Stage (LightGBM),$558.55,0.2204,80.92%,94.52%
3,BTYD Statistical Baseline,$515.40,0.4786,74.67%,130.84%
4,XGBoost,$526.44,0.3888,76.27%,161.87%
5,Ridge (L2),"$9,641.97",-13812.7760,1396.94%,165.19%
6,Random Forest,$507.88,0.5131,73.58%,163.50%
7,Linear Regression,"$22,771.41",-85356.1691,3299.15%,165.24%
8,ElasticNet,"$1,114.04",-20.7669,161.40%,165.44%
9,LightGBM,$535.97,0.3652,77.65%,159.46%



=== SEGMENT-LEVEL PERFORMANCE ===


,Segment,N,RMSE,MAE,R2,WAPE%,SMAPE%,Avg_Actual,Avg_Pred
0,Top 20% Spenders,76,"$3,615.14","$2,348.31",0.3576,58.61%,93.83%,"$4,006.50","$2,023.94"
1,Mid Spenders,236,$564.99,$438.29,-2.0512,67.71%,107.27%,$647.30,$391.94
2,Low Spenders,61,$178.70,$137.80,-9.6733,97.27%,134.64%,$141.66,$129.88
3,Zero-Spend (Churned),302,$258.07,$103.73,0.0000,nan%,49.67%,$0.00,$103.73
4,All Customers,675,"$1,271.14",$476.50,0.5755,69.04%,82.46%,$690.22,$423.06
